
# DCGAN for MNIST

In this jupyter notebook we train a convolutional generative adversarial network (DCGAN) to generate handwritten digits. 

Note: this code (for cpu) can be easily extended to RGB images.

## DCGAN

A DCGAN is a direct extension of the GAN described above, except that it
explicitly uses convolutional and convolutional-transpose layers in the
discriminator and generator, respectively. The discriminator
is made up of strided
[convolution](https://pytorch.org/docs/stable/nn.html#torch.nn.Conv2d)_
layers, [batch
norm](https://pytorch.org/docs/stable/nn.html#torch.nn.BatchNorm2d)_
layers, and
[LeakyReLU](https://pytorch.org/docs/stable/nn.html#torch.nn.LeakyReLU)_
activations. The input is a 1x28x28 input image and the output is a
scalar probability that the input is from the real data distribution.
The generator is comprised of
[convolutional-transpose](https://pytorch.org/docs/stable/nn.html#torch.nn.ConvTranspose2d)_
layers, batch norm layers, and
[ReLU](https://pytorch.org/docs/stable/nn.html#relu)_ activations. The
input is a latent vector, $z$, that is drawn from a standard
normal distribution and the output is a 1x28x28 image. The strided
conv-transpose layers allow the latent vector to be transformed into a
volume with the same shape as an image.


Note: the bias are set to False because BN eliminates the bias (we can keep True but it increase the number of training parameters), and BN (with affine=True) adds a bias (the beta term in batch norm adds a bias to each channel).


## Import the necessary libraries

In [ ]:
import torch
from torch import nn

import math
import matplotlib.pyplot as plt
import matplotlib.animation as animation

import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils

import numpy as np

from IPython.display import HTML

## Inputs

Let’s define some inputs for the run:

-  ``dataroot`` - the path to the root of the dataset folder. We will
   talk more about the dataset in the next section.
-  ``batch_size`` - the batch size used in training. The DCGAN paper
   uses a batch size of 128.
-  ``image_size`` - the spatial size of the images used for training.
-  ``nc`` - number of color channels in the input images.
-  ``nz`` - length of latent vector.
-  ``ngf`` - relates to the depth of feature maps carried through the
   generator.
-  ``ndf`` - sets the depth of feature maps propagated through the
   discriminator.
-  ``num_epochs`` - number of training epochs to run. Training for
   longer will probably lead to better results but will also take much
   longer.
-  ``lr`` - learning rate for training.
-  ``beta1`` - beta1 hyperparameter for Adam optimizers (see [torch.optim.Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html)).


In [ ]:
# Root directory for dataset
dataroot = "data"

# Batch size during training
batch_size = 32

# Spatial size of training images. All images will be resized to this
#   size using a transformer.
image_size = 28

# Number of channels in the training images. For color images this is 3
nc = 1

# Size of z latent vector (i.e. size of generator input)
nz = 100

# Size of feature maps in generator
ngf = 32

# Size of feature maps in discriminator
ndf = 28 # proportinal to the input image size

# Number of training epochs
num_epochs = 10

# Learning rate for optimizers (according to the original DCGAN paper)
lr = 0.0002

# Beta1 hyperparameter for Adam optimizers (according to the original DCGAN paper)
beta1 = 0.5

## Training Data
The MNIST dataset consists of 1x28×28 pixel grayscale images of handwritten digits from 0 to 9. The database contains 60,000 training images and 10,000 testing images. To use them with PyTorch, you’ll need to perform some conversions. For that, we define transform, a function to be used when loading the data.

In [ ]:
# transform funtion
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)

# Get MNIST dataset
train_set = torchvision.datasets.MNIST(root=dataroot, train=True, transform=transform, download=True)

# Create the dataloader
train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True, drop_last=True)

# Plot some training images
real_batch = next(iter(train_loader))
plt.figure(figsize=(8,8))
plt.axis("off")
plt.title("Training Images")
plt.imshow(np.transpose(vutils.make_grid(real_batch[0][:64], padding=2, normalize=True),(1,2,0)))
plt.show()

## Implementation

### Weight Initialization

From the DCGAN paper, the authors specify that all model weights shall
be randomly initialized from a Normal distribution with ``mean=0``,
``stdev=0.02``. The ``weights_init`` function takes an initialized model as
input and reinitializes all convolutional, convolutional-transpose, and
batch normalization layers to meet this criteria. This function is
applied to the models immediately after initialization.


In [ ]:
# custom weights initialization
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        torch.nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm2d") != -1:
        torch.nn.init.normal_(m.weight.data, 1.0, 0.02)
        torch.nn.init.constant_(m.bias.data, 0.0)

### Implementing the Generator

The generator, $G$, is designed to map the latent space vector
($z$) to data-space. Since our data are images, converting
$z$ to data-space means ultimately creating a image with the
same size as the training images (i.e. here 1x28x28). In practice, this is
accomplished through a series of strided two dimensional convolutional
transpose layers, each paired with a 2D batch norm layer and a relu
activation. The output of the generator is fed through a tanh function
to return it to the input data range of $[-1,1]$. It is worth
noting the existence of the batch norm functions after the
conv-transpose layers, as this is a critical contribution of the DCGAN
paper. These layers help with the flow of gradients during training. 

Notice, how the inputs we set in the input section (``nz``, ``ngf``, and
``nc``) influence the generator architecture in code. ``nz`` is the length
of the z input vector, ``ngf`` relates to the size of the feature maps
that are propagated through the generator, and ``nc`` is the number of
channels in the output image (set to 1 for gray level images). 


Recalls:
 - torch.nn.BatchNorm2d(num_features, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, device=None, dtype=None)
 - torch.nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride=1, padding=0, output_padding=0, groups=1, bias=True, dilation=1, padding_mode='zeros', device=None, dtype=None)
 - torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True, padding_mode='zeros', device=None, dtype=None)



In [ ]:
# Generator Code
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(nz, ngf*4, kernel_size = 3, stride = 2, padding = 0, bias=False),
            nn.BatchNorm2d(ngf*4),
            nn.ReLU(True),
            # state size. (ngf*4) x 3 x 3
            nn.ConvTranspose2d(ngf*4, ngf*2, kernel_size = 3, stride = 2, padding = 0, bias=False),
            nn.BatchNorm2d(ngf*2),
            nn.ReLU(True),
            # state size. (ngf*2) x 8 x 8
            nn.ConvTranspose2d(ngf*2, ngf, kernel_size = 3, stride = 2, padding = 0, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. (ngf) x 16 x 16
            nn.ConvTranspose2d(ngf, nc, kernel_size = 3, stride = 2, padding = 2, output_padding = 1, bias=False),
            nn.Tanh()
            # state size. (nc) x 28 x 28
        )

    def forward(self, input):
        return self.model(input)

Now, we can instantiate the generator and apply the ``weights_init``
function. Check out the printed model to see how the generator object is
structured.




In [ ]:
# Create the generator
generator = Generator()

# Apply the ``weights_init`` function to randomly initialize all weights
generator.apply(weights_init)

# Visualize the model
print(generator)

### Discriminator

The discriminator, $D$, is a binary classification
network that takes an image as input and outputs a scalar probability
that the input image is real (as opposed to fake). Here, $D$ takes
a 1x28x28 input image, processes it through a series of Conv2d,
BatchNorm2d, and LeakyReLU layers, and outputs the final probability
through a Sigmoid activation function. This architecture can be extended
with more layers if necessary for the problem, but there is significance
to the use of the strided convolution, BatchNorm, and LeakyReLUs. The
DCGAN paper mentions it is a good practice to use strided convolution
rather than pooling to downsample because it lets the network learn its
own pooling function. Also batch norm and leaky relu functions promote
healthy gradient flow which is critical for the learning process of both
$G$ and $D$.


In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            # input is (nc) x 28 x 28
            nn.Conv2d(nc, ndf, kernel_size = 4, stride = 2, padding = 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf) x 14 x 14
            nn.Conv2d(ndf, ndf*2, kernel_size = 4, stride = 2, padding = 1, bias=False),
            nn.BatchNorm2d(ndf*2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*2) x 7 x 7
            nn.Conv2d(ndf*2, ndf*4, kernel_size = 4, stride = 2, padding = 1, bias=False),
            nn.BatchNorm2d(ndf*4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*4) x 3 x 3
            nn.Conv2d(ndf*4, 1, kernel_size = 4, stride = 2, padding = 1, bias=False),
            nn.Sigmoid() 
        )

    def forward(self, input):
        return self.model(input)

Now we can create the discriminator, apply the
``weights_init`` function, and print the model’s structure.




In [ ]:
# Create the Discriminator
discriminator = Discriminator()

# Apply the ``weights_init`` function to randomly initialize all weights
discriminator.apply(weights_init)

# Print the model
print(discriminator)

### Loss Functions and Optimizers

With $D$ and $G$ setup, we can specify how they learn
through the loss functions and optimizers. We will use the Binary Cross
Entropy loss
([BCELoss](https://pytorch.org/docs/stable/generated/torch.nn.BCELoss.html#torch.nn.BCELoss)_)
function.

We define our real label as 1 and the fake label as 0. These labels will be used when calculating the losses of $D$ and $G$. 

Finally, we set up two separate optimizers, one for $D$ and
one for $G$. As specified in the DCGAN paper, both are Adam
optimizers with learning rate 0.0002 and Beta1 = 0.5. For keeping track
of the generator’s learning progression, we will generate a fixed batch
of latent vectors that are drawn from a Gaussian distribution
(i.e. fixed_noise) . In the training loop, we will periodically input
this fixed_noise into $G$, and over the iterations we will see
images form out of the noise.


In [ ]:
# Initialize the ``BCELoss`` function
criterion = nn.BCELoss()

# Create batch of latent vectors that we will use to visualize
#  the progression of the generator
fixed_noise = torch.randn(64, nz, 1, 1)

# Establish convention for real and fake labels during training
real_label = 1
fake_label = 0

# Setup Adam optimizers for both G and D
optimizer_discriminator = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))
optimizer_generator = torch.optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))

## Training loop

Recall of best practices:
- Discriminator: construct different mini-batches for real and fake 
- Generator: adjust G’s objective function to maximize $log(D(G(z)))$ (non-saturing loss)
- Training is split up into two main parts: Part 1 updates the Discriminator and Part 2 updates the Generator.


**Part 1 - Train the Discriminator**

The goal of training the Discriminator is to maximize the probability of correctly classifying a given input as real or fake. Practically, we want to maximize $log(D(x)) + log(1-D(G(z)))$. Due to the separate mini-batch
suggestion, we calculate this in two steps: 
 - Firstly, we construct a batch of real samples from the training set, forward
pass through $D$, calculate the loss ($log(D(x))$), then we calculate the gradients in a backward pass.
 -  Secondly, we construct a batch of fake samples with the current Generator, forward pass this
batch through $D$, calculate the loss ($log(1-D(G(z)))$), and *accumulate* the gradients with a backward pass.

Finally, with the gradients accumulated from both the all-real and all-fake batches, we call a step of the Discriminator’s optimizer.

**Part 2 - Train the Generator**

As stated in the original paper by Goodfellow, we train the Generator by
maximizing $log(D(G(z)))$, to avoid vanishing gradients. In the code we accomplish
this by: classifying the Generator output from Part 1 with the
Discriminator, computing G’s loss *using real labels as ground truth (GT)*, computing
G’s gradients in a backward pass, and finally updating G’s parameters
with an optimizer step. Using the real labels as GT labels for the loss function allows us to use the
$log(x)$ part of the ``BCELoss`` (rather than the $log(1-x)$
part), which is exactly what we want.

Finally, we will do some statistic reporting and at the end of each
epoch we will push our fixed_noise batch through the generator to
visually track the progress of G’s training. The training statistics
reported are:

-  **Loss_D** - discriminator loss calculated as the sum of losses for
   the all real and all fake batches ($log(D(x)) + log(1 - D(G(z)))$).
-  **Loss_G** - generator loss calculated as $log(D(G(z)))$
-  **D(x)** - the average output (across the batch) of the discriminator
   for the all real batch. This should start close to 1 then
   theoretically converge to 0.5 when G gets better.
-  **D(G(z))** - average discriminator outputs for the all fake batch.
   The first number is before D is updated and the second number is
   after D is updated. These numbers should start near 0 and converge to
   0.5 as G gets better.



In [ ]:
# Set models to train mode
generator.train()
discriminator.train()

# Lists to keep track of progress
img_list = []
G_losses = []
D_losses = []
iters = 0


# For each epoch
for epoch in range(num_epochs):
    
    # For each batch in the dataloader
    for i, (real_samples, mnist_labels) in enumerate(train_loader):
        
        ############################
        # (1) Update Discriminator
        ###########################
        discriminator.zero_grad()
        
        ## Train with real batch
        real_samples = real_samples
        real_samples_labels = torch.ones((batch_size,))# we don't take into account the labels...
        output_real = discriminator(real_samples).view(-1)
        D_x = output_real.mean().item()

        # Calculate loss on all-real batch
        errD_real = criterion(output_real, real_samples_labels)
        
        ## Train with fake batch
        latent_space_samples = torch.randn((batch_size, nz, 1, 1)) 
        with torch.no_grad():
            generated_samples = generator(latent_space_samples)
        generated_samples_labels = torch.zeros((batch_size,))
        output_fake = discriminator(generated_samples).view(-1)
        D_G_z1 = output_fake.mean().item()
        
        # Calculate loss on all-fake batch
        errD_fake = criterion(output_fake, generated_samples_labels)
        
        # Compute error of D as sum over the fake and the real batches
        errD = (errD_real + errD_fake)/2
        
        # Calculate the gradients 
        errD.backward()
     
        # Update D
        optimizer_discriminator.step()

        ############################
        # (2) Update Generator
        ###########################
        # Data for training the generator
        generator.zero_grad()
        latent_space_samples = torch.randn((batch_size, nz, 1, 1))

        # Training the generator
        generated_samples = generator(latent_space_samples)
        output_discriminator_generated = discriminator(generated_samples).view(-1)
        D_G_z2 =output_discriminator_generated.mean().item()
        
        errG = criterion(output_discriminator_generated, real_samples_labels)
        
        # Calculate gradients for Generator
        errG.backward()
        
        # Update Generator
        optimizer_generator.step()
        
        # Output training stats
        if i % 100 == 0:
            print('[%d/%d][%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f\tD(x): %.4f\tD(G(z)): %.4f / %.4f'
                  % (epoch+1, num_epochs, i, len(train_loader),
                     errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))
        
        # Save Losses for plotting later
        G_losses.append(errG.item())
        D_losses.append(errD.item())
        
        # Check how the generator is doing by saving G's output on fixed_noise
        if (iters % 500 == 0) or ((epoch == num_epochs-1) and (i == len(train_loader)-1)):
            generator.eval()
            with torch.no_grad():
                fake = generator(fixed_noise) # send the data to cpu
            img_list.append(vutils.make_grid(fake, padding=2, normalize=True))
            generator.train()
        iters += 1

## Results

**Loss versus training iteration**

Plot of D & G’s losses versus training iterations.


In [ ]:
plt.figure(figsize=(10,5))
plt.title("Generator and Discriminator Loss During Training")
plt.plot(G_losses,label="G")
plt.plot(D_losses,label="D")
plt.xlabel("iterations")
plt.ylabel("Loss")
plt.legend()
plt.show()

**Visualization of G’s progression**

Visualize G’s output on the fixed_noise batch for every epoch with an animation. Press the play button to start the
animation.


In [ ]:
fig = plt.figure(figsize=(8,8))
plt.axis("off")
ims = [[plt.imshow(np.transpose(i,(1,2,0)), animated=True)] for i in img_list]
ani = animation.ArtistAnimation(fig, ims, interval=1000, repeat_delay=1000, blit=True)

HTML(ani.to_jshtml())

**Real Images vs. Fake Images**

Look at some real images and fake images side by side.


In [ ]:
# Grab a batch of real images from the dataloader
real_batch = next(iter(train_loader))

# Plot the real images
plt.figure(figsize=(15,15))
plt.subplot(1,2,1)
plt.axis("off")
plt.title("Real Images")
plt.imshow(np.transpose(vutils.make_grid(real_batch[0][:64], padding=5, normalize=True),(1,2,0)))

# Plot the fake images from the last epoch
plt.subplot(1,2,2)
plt.axis("off")
plt.title("Fake Images")
plt.imshow(np.transpose(img_list[-1],(1,2,0)))
plt.show()

## Generate new data

In [ ]:
# Turn generator to eval mode
generator.eval()

# Generate new data of latent vectors
new_noise = torch.randn(20, nz, 1, 1)
new_fake = generator(new_noise)

# Plot the new fake images from the last epoch
plt.figure(figsize=(5,5))
plt.axis("off")
plt.title("Real Images")
plt.imshow(np.transpose(vutils.make_grid(new_fake, nrow=5, normalize=True),(1,2,0)))